# BG-forecasting — cold-start history budgets on Colab

> **Sibling notebook.** `run_cold_start_on_colab.ipynb` runs the same experiment
> from an embedded snapshot of the runner and writes to
> `<DRIVE_RESULTS>/cold_start_followup/`. This one drives the runner from the
> repository checkout instead, in the style of `run_on_colab.ipynb`, and adds
> two things the other notebook does not have: the **TL-vs-RL figure**
> (section 8) and the **screening re-test** (section 9). If
> you have already run the other one, set `RUN_TRAINING = False` in section 6 and
> point it at that output — sections 1-7 then cost nothing and section 8 still
> works.

Runs `RUN/experiments/run_cold_start.py`: the same twelve patients and three
seeds as the publication grid, but with the target patient's training history
cut back to **1, 3, 7 days and full**, while the official test windows stay
fixed.

### What this is for

The article measures transfer with six weeks of target data and finds a
1.18-3.64% MAE reduction. Two review points hang off that number:

* **R1 W1** treats the moderate benefit as a weakness. If the benefit grows as
  the history shrinks, the published figure is a lower bound for the setting
  where transfer is actually needed, and the limitation section's cold-start
  paragraph stops being a promise and becomes a result.
* **The difficulty/screening contribution is currently null.** With six weeks of
  data the per-patient TL-RL differences are a few tenths of a mg/dL, so there
  is almost no signal for a screen to predict. If the benefit at one day is
  large and spread unevenly across patients, section 8 below re-tests whether
  shift or signal irregularity predicts *who* benefits. That is the question the
  article currently answers with "these exploratory results do not establish who
  should receive transfer".

Any positive result here is a claim about the **cold-start regime only**. It
does not reinstate a screen for the six-week setting, and it cannot explain why
the submitted numbers changed.

---

### Read before running

**The dataset is DUA-restricted.** OhioT1DM is governed by a data use
agreement. This notebook expects a copy in your Drive, which places it on
third-party storage. Check your agreement permits that before uploading
anything.

**Budget 2-4 h on a T4 for the full grid**, dominated by the 36 leave-target-out
pre-trainings (12 patients x 3 seeds). Everything is mirrored to Drive after
each job and completed jobs are skipped by SHA-checked markers, so a disconnect
costs at most one job. Run the smoke configuration first.

**Pick a GPU runtime.** `run()` raises rather than falling back to CPU.

**This experiment has its own protocol** and does not reuse the article's
training schedule: chronological validation inside the budget, a 50-epoch
target cap with patience 10, and source pre-training validated on source
patients only. Compare budgets *within* this run. Do not swap its best budget in
for the published full-history headline.


## 1 · Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- edit these to match your Drive ---------------------------------------
DRIVE_DATA    = "/content/drive/MyDrive/ohiot1dm"      # must contain 2018/ and 2020/
DRIVE_RESULTS = "/content/drive/MyDrive/bg-results"    # results are mirrored here
REPO_DIR      = "/content/BG-forecasting"

# Where the code comes from.
#   "git"   -> clone REPO_URL at BRANCH
#   "drive" -> copy DRIVE_REPO (use this if the branch is not pushed)
SOURCE     = "git"
REPO_URL   = "https://github.com/beatriz-fulgencio/BG-forecasting.git"
BRANCH     = "bench2"
DRIVE_REPO = "/content/drive/MyDrive/BG-forecasting"
# ---------------------------------------------------------------------------

import os, pathlib

# Separate from the grid's results/ tree: nothing here feeds the published
# tables, and mixing the two invites reading a cold-start MAE as a headline one.
COLD_LOCAL = f"{REPO_DIR}/results/cold_start"
COLD_DRIVE = f"{DRIVE_RESULTS}/cold_start"
for d in (DRIVE_RESULTS, COLD_DRIVE, f"{DRIVE_RESULTS}/logs"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Drive ready. Cold-start outputs mirror to", COLD_DRIVE)

## 2 · Get the code

`RUN/experiments/run_cold_start.py` is the whole experiment: data slicing,
training, evaluation, bootstrap and report. This notebook only feeds it paths
and a config.

In [ ]:
import shutil, subprocess, sys, pathlib

if SOURCE == "git":
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("WARNING: git pull --ff-only failed, so this checkout may be stale.\n"
                  + (pull.stderr or pull.stdout).strip() + "\n")
elif SOURCE == "drive":
    if not pathlib.Path(REPO_DIR).is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR)
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

RUNNER  = pathlib.Path(REPO_DIR) / "RUN" / "experiments" / "run_cold_start.py"
SUPPORT = pathlib.Path(REPO_DIR) / "benchmark" / "data"
missing = [str(p) for p in (RUNNER, SUPPORT / "loaders.py", SUPPORT / "preprocessors.py")
           if not p.is_file()]
if missing:
    raise SystemExit("Missing from this checkout:\n  " + "\n  ".join(missing) +
                     "\n\nCommit and push them, or set SOURCE = 'drive'.")
print("Runner:", RUNNER)

## 3 · Stage the OhioT1DM data

In [ ]:
import shutil, pathlib

# 6 patients x 2 releases x train+test = 24 XML files.
COHORT = {"2018": [559, 563, 570, 575, 588, 591],
          "2020": [540, 544, 552, 567, 584, 596]}

dst = pathlib.Path(REPO_DIR) / "data" / "raw" / "ohiot1dm"
src = pathlib.Path(DRIVE_DATA)
if not src.is_dir():
    raise SystemExit(f"{DRIVE_DATA} not found. Upload your OhioT1DM copy there first.")

# File by file: skipping a release whose directory already exists would leave an
# interrupted copy permanently incomplete.
copied = 0
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        (dst / release / mode).mkdir(parents=True, exist_ok=True)
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            target, source = dst / release / mode / name, src / release / mode / name
            if target.is_file() or not source.is_file():
                continue
            shutil.copy2(source, target)
            copied += 1
print(f"{copied} file(s) copied from Drive.")

missing = []
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            if not (dst / release / mode / name).is_file():
                where = "absent from Drive too" if not (src / release / mode / name).is_file() else "copy failed"
                missing.append(f"{release}/{mode}/{name} ({where})")
if missing:
    raise SystemExit("Missing data files:\n  " + "\n  ".join(missing))
print("All 24 XML files staged.")

## 4 · Check the runtime

`run()` refuses to start on CPU when `device='cuda'` rather than quietly running
for a day at CPU speed.

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("\nNo GPU. Runtime > Change runtime type > T4 GPU, then rerun this cell.")

## 5 · Configure the run

`SMOKE = True` runs a deliberately tiny version — four patients, one seed, two
budgets, two epochs — to prove the plumbing works end to end. Its outputs are
named `SMOKE_*` and its report says so on the first line. **It is not evidence.**

Flip to `False` for the real grid. Nothing else needs changing: the defaults in
`run_cold_start.py` already carry the twelve patients, seeds 41-43 and the
1/3/7/full budgets.

In [ ]:
import importlib.util, copy, json

spec = importlib.util.spec_from_file_location("run_cold_start", str(RUNNER))
cold = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cold)

SMOKE = True          # <-- set False for the real run

CFG = copy.deepcopy(cold.DEFAULTS)
CFG.update(model="gru", horizon_steps=6)      # GRU, 30 min: the article's headline cell

if SMOKE:
    CFG.update(smoke=True, patients=[540, 544, 552, 559], seeds=[41],
               budgets_days=[1, "full"], source_epochs=1, target_epochs=3,
               bootstrap_replicates=500)

print(json.dumps({k: v for k, v in CFG.items()
                  if k in ("model", "horizon_steps", "budgets_days", "seeds", "patients",
                           "source_epochs", "target_epochs", "target_patience",
                           "validation_fraction", "bootstrap_replicates", "smoke")}, indent=2))
print("\nMode:", "SMOKE - not evidence" if CFG["smoke"] else "FULL RUN")

## 6 · Run

Every patient and budget is preflighted before any GPU time is spent, so a
patient whose history is shorter than a requested budget fails the whole run up
front instead of being silently dropped — eligibility cannot end up selected by
performance. Completed jobs are restored from Drive and skipped, so rerunning
this cell after a disconnect picks up where it stopped.

In [ ]:
import pathlib, shutil

RUN_TRAINING = True    # False: reuse a finished run instead of training again
EXISTING_RUN = ""      # e.g. "/content/drive/MyDrive/bg-results/cold_start_followup/gru_30min_<fingerprint>"

if RUN_TRAINING:
    local, remote = cold.run(data_root=f"{REPO_DIR}/data",
                             support=str(SUPPORT),
                             cfg=CFG,
                             local_root=COLD_LOCAL,
                             drive_root=COLD_DRIVE,
                             device=DEVICE)
else:
    # Reuse whatever either notebook produced. The summary directory is the
    # contract; anything without it is an unfinished run, not a reusable one.
    remote = pathlib.Path(EXISTING_RUN)
    if not (remote / "summary" / "patient_seed_metrics.csv").is_file():
        raise SystemExit(
            f"No finished run at {remote}.\n"
            "Point EXISTING_RUN at a directory containing summary/patient_seed_metrics.csv "
            "(look under <DRIVE_RESULTS>/cold_start/ or <DRIVE_RESULTS>/cold_start_followup/), "
            "or set RUN_TRAINING = True.")
    local = pathlib.Path(COLD_LOCAL) / remote.name
    shutil.copytree(remote, local, dirs_exist_ok=True)
    # Section 8 iterates CFG["budgets_days"]; take them from the run itself so a
    # reused run is not silently analysed against this notebook's defaults.
    import json as _json
    CFG["budgets_days"] = _json.loads((local / "protocol.json").read_text())["config"]["budgets_days"]
    print("Reusing", remote, "\nbudgets:", CFG["budgets_days"])

print("\nLocal :", local)
print("Drive :", remote)

## 7 · The history-budget curve

`benefit` is RL error minus TL error, so positive means transfer helped.
`additional_benefit_vs_full` is the extra benefit at that budget over this
experiment's own full-history control, with intervals from the same joint
patient-and-seed draws — that difference, not the raw percentage, is what
answers "does transfer help more when there is less data".

In [ ]:
import pandas as pd, pathlib
from IPython.display import Image, display

summary = pathlib.Path(local) / "summary"
table = pd.read_csv(summary / "budget_summary.csv")

cols = ["metric", "budget_days", "regular_mean", "transfer_mean", "benefit",
        "benefit_ci_low", "benefit_ci_high", "relative_benefit_pct",
        "relative_ci_low", "relative_ci_high", "additional_benefit_vs_full",
        "additional_ci_low", "additional_ci_high", "wilcoxon_p", "wilcoxon_bh_q"]
print(table[cols].to_string(index=False, float_format=lambda v: f"{v:8.3f}"))

print("\n" + (summary / "REPORT.md").read_text())
display(Image(filename=str(summary / "history_budget_curve.png")))

## 8 · TL vs RL across the history budgets

The figure the cold-start question is actually asking for: both regimes at every
budget, on one axis.

* **(a)** absolute error for RL and TL with the persistence anchor, so a large
  relative gain on a model that is itself worse than a no-change forecast cannot
  be mistaken for a good result;
* **(b)** the paired RL$-$TL difference with its interval;
* **(c)** the same benefit per patient, because a cohort mean carried by two
  patients is what a screening claim would rest on — and section 9 then tests
  whether anything predicts which patients those are.

Intervals come from the same joint patient-and-seed resampling as the rest of
the paper, with the same draws applied to both regimes. The CSV beside the
figure adds `additional_vs_control`: each budget's benefit minus the
full-history benefit under identical draws, which is the paired form of "does
transfer help more when there is less data".

`RUN/experiments/plot_cold_start_tl_vs_rl.py` takes any finished run, so it also
draws the output of `run_cold_start_on_colab.ipynb`:

```
python RUN/experiments/plot_cold_start_tl_vs_rl.py --run-dir <run> [--metric rmse]
```

In [ ]:
import subprocess, sys, pathlib, shutil
from IPython.display import Image, display

METRIC = "mae"          # "rmse" for the same figure on squared-error terms

result = subprocess.run(
    [sys.executable, "RUN/experiments/plot_cold_start_tl_vs_rl.py",
     "--run-dir", str(local), "--metric", METRIC,
     "--bootstrap-count", str(CFG["bootstrap_replicates"]),
     "--resampling-seed", "42", "--pdf"],
    cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise SystemExit(f"Figure failed:\n{result.stderr}")

produced = pathlib.Path(local) / "summary"
for name in (f"cold_start_tl_vs_rl_{METRIC}.png",
             f"cold_start_tl_vs_rl_{METRIC}.pdf",
             f"cold_start_tl_vs_rl_{METRIC}.csv"):
    shutil.copy2(produced / name, pathlib.Path(remote) / "summary" / name)

display(Image(filename=str(produced / f"cold_start_tl_vs_rl_{METRIC}.png")))

## 9 · Does a screen work when the benefit is large?

This is the part that could revive the paper's second contribution.

In the published six-week setting the per-patient TL-RL differences are a few
tenths of a mg/dL, and neither shift nor entropy predicts them
(ρ = −0.105 to 0.385, all p > 0.2). That is a test with almost nothing to
detect. If the one-day benefit is several mg/dL and uneven across patients,
the same screens get a real target.

Two targets are tested, both as per-patient cross-seed means within each budget:

* `benefit_pct` — relative transfer benefit, i.e. *who should receive transfer*;
* `transfer_mae` — residual difficulty under transfer, i.e. *who stays hard*.

Screens are labelled by when they can be computed. Only a **training-only**
screen could ever run before the held-out period exists, so only those can
support the advance screening the paper says remains unvalidated. `shift_score`
and test-period `sample_entropy` are retrospective and are reported for
comparison with the published analysis, not as deployable screens.

n = 12 patients per budget. Permutation tests shuffle whole patients;
Benjamini-Hochberg runs across all budget x screen tests within each target.

In [ ]:
import pathlib, shutil, numpy as np, pandas as pd
from scipy.stats import rankdata

FEATURES_LOCAL = pathlib.Path(REPO_DIR) / "results/analysis/dataset/dataset_signal_features.csv"
FEATURES_DRIVE = pathlib.Path(DRIVE_RESULTS) / "analysis/dataset/dataset_signal_features.csv"
if not FEATURES_LOCAL.is_file() and FEATURES_DRIVE.is_file():
    FEATURES_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FEATURES_DRIVE, FEATURES_LOCAL)
if not FEATURES_LOCAL.is_file():
    raise SystemExit(
        "dataset_signal_features.csv not found locally or in Drive.\n"
        "Produce it once with the signal stage of the dataset analysis — section 10 of\n"
        "run_on_colab.ipynb, or:  bash RUN/run_dataset_analysis.sh signal\n"
        "(it needs one completed grid run to read the cohort settings from).")

SCREENS = {"train_sample_entropy": "training-only",
           "train_autocorr_lag1":  "training-only",
           "shift_score":          "needs held-out period",
           "sample_entropy":       "needs held-out period"}
PERMUTATIONS = 20000
rng = np.random.default_rng(20260923)

metrics = pd.read_csv(pathlib.Path(local) / "summary" / "patient_seed_metrics.csv")
metrics["benefit"] = metrics.regular_mae - metrics.transfer_mae
metrics["benefit_pct"] = 100 * metrics.benefit / metrics.regular_mae
per_patient = (metrics.groupby(["budget_days", "patient_id"], as_index=False)
                      [["benefit", "benefit_pct", "transfer_mae", "regular_mae"]].mean())

features = pd.read_csv(FEATURES_LOCAL)[["patient_id", *SCREENS]]
joined = per_patient.merge(features, on="patient_id", how="left", validate="many_to_one")
if joined[list(SCREENS)].isna().any().any():
    raise SystemExit("Some patients have no signal features; regenerate dataset_signal_features.csv.")


def spearman_permutation(x, y, permutations=PERMUTATIONS):
    """Spearman rho with a whole-patient permutation p-value.

    Permuting the screen across patients is the exact null for n=12: it keeps
    each patient's outcome intact and only breaks the pairing.
    """
    rx, ry = rankdata(x) - np.mean(rankdata(x)), rankdata(y) - np.mean(rankdata(y))
    scale = np.sqrt((rx ** 2).sum() * (ry ** 2).sum())
    observed = float(rx @ ry / scale)
    null = np.tile(rx, (permutations, 1))
    null = rng.permuted(null, axis=1) @ ry / scale
    p = (1 + int((np.abs(null) >= abs(observed) - 1e-12).sum())) / (permutations + 1)
    return observed, p


def benjamini_hochberg(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    q = np.empty_like(p)
    q[order] = np.minimum.accumulate(
        (p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    return np.minimum(q, 1.0)


rows = []
for target in ("benefit_pct", "transfer_mae"):
    for budget in CFG["budgets_days"]:
        group = joined[joined.budget_days == str(budget)]
        for screen, availability in SCREENS.items():
            rho, p = spearman_permutation(group[screen].to_numpy(float),
                                          group[target].to_numpy(float))
            rows.append(dict(target=target, budget_days=str(budget), screen=screen,
                             availability=availability, n_patients=len(group),
                             spearman_rho=rho, permutation_p=p))

screen_table = pd.DataFrame(rows)
screen_table["bh_q"] = np.concatenate(
    [benjamini_hochberg(g.permutation_p.to_numpy())
     for _, g in screen_table.groupby("target", sort=False)])
screen_table["survives_bh_0.05"] = screen_table.bh_q < 0.05

out = pathlib.Path(local) / "summary" / "screen_vs_benefit.csv"
screen_table.to_csv(out, index=False)
shutil.copy2(out, pathlib.Path(remote) / "summary" / out.name)

pd.set_option("display.width", 160)
print(screen_table.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))
print("\nSaved:", out)

### Read the screen table this way

* **Nothing survives BH.** The paper's current wording stands unchanged, and the
  cold-start run has simply widened the setting in which the screen fails. Say
  so in the limitations: the null is not an artefact of the six-week data-rich
  condition.
* **A training-only screen survives at the short budgets.** That is a real,
  reportable result and it restores the second contribution — but scoped:
  *in the cold-start regime, irregularity measurable before training identifies
  who gains least from transfer.* It says nothing about the six-week setting,
  where the same screen is null.
* **Only `shift_score` or test-period `sample_entropy` survives.** Interesting
  but not deployable, and it must be labelled retrospective. It would, however,
  give the shift indicator a defensible role again.

At n = 12 a single influential patient moves ρ a long way. Before writing any of
this up, plot the twelve points per surviving cell and check it is not one
patient doing the work.

In [ ]:
import matplotlib.pyplot as plt

survivors = screen_table[screen_table["survives_bh_0.05"]]
if survivors.empty:
    print("No cell survives BH correction. Nothing to inspect; the null holds "
          "in the cold-start regime too.")
else:
    fig, axes = plt.subplots(1, len(survivors), figsize=(4.2 * len(survivors), 3.8),
                             squeeze=False, constrained_layout=True)
    for ax, (_, row) in zip(axes[0], survivors.iterrows()):
        group = joined[joined.budget_days == row.budget_days]
        ax.scatter(group[row.screen], group[row.target])
        for _, point in group.iterrows():
            ax.annotate(int(point.patient_id), (point[row.screen], point[row.target]),
                        fontsize=7, xytext=(3, 3), textcoords="offset points")
        ax.set(xlabel=row.screen, ylabel=row.target,
               title=f"{row.budget_days}d: rho={row.spearman_rho:.2f}, q={row.bh_q:.3f}")
    plt.show()

## 10 · Where the outputs are

```
<DRIVE_RESULTS>/cold_start/<model>_<horizon>min_<fingerprint>/
    protocol.json                 config, input file hashes, library versions
    preflight.json                per-patient/budget window audits
    pretraining/                  leave-target-out source checkpoints
    jobs/patient<ID>_seed<S>_days<B>/
        regular_predictions.csv   per-window truth/prediction/persistence
        transfer_predictions.csv
        *_history.json            per-epoch train/validation MSE
        complete.json             metrics + SHA markers that drive the skipping
    summary/
        patient_seed_metrics.csv  one row per patient x seed x budget
        budget_summary.csv        the history-budget curve, with intervals
        cold_start_tl_vs_rl_mae.png    section 8 (+ .pdf and .csv)
        screen_vs_benefit.csv          section 9
        history_budget_curve.png       the runner's own relative-benefit curve
        REPORT.md
```

The directory name ends in a fingerprint of the config, input files and runner
source, so changing any of them starts a clean directory instead of mixing
results. Two runs that differ in anything that matters cannot land in the same
folder.

### If this goes into the paper

The cold-start result belongs in its own short paragraph, not folded into the
headline. The honest framing is: *the published 1.18-3.64% describes a
data-rich six-week setting; at a one-day budget under a separate protocol the
benefit is X%, so the published figure is a lower bound for newly monitored
patients* — which is what the conclusion currently offers as an open question.
The protocol differences listed in `REPORT.md` have to travel with the number.